# Data Preprocessing — Final
### Machine Learning-Based IDS — CICIDS2017 → Suricata Deployment

**PURPOSE OF THIS NOTEBOOK:**
This notebook transforms the raw CICIDS2017 dataset into clean training and test sets
ready for the Random Forest model. It also saves all artifacts needed by the inference
pipeline (extractor.py + detector.py) at deployment time.

**KEY DECISIONS MADE HERE:**
- 6 classes: BENIGN, DoS (incl. DDoS), PortScan, Brute Force, Bot, Web Attack
- Heartbleed (11 samples) and Infiltration (36 samples) dropped — too few to learn from
- 19 Suricata-aligned features (bwd_psh_flags was constant across all rows → dropped)
- No SMOTE — class imbalance handled via class_weight='balanced' in Random Forest
- Evaluation metric: **macro F1** — treats all 6 classes equally regardless of size

**OUTPUTS (saved at end of notebook):**
| File | Purpose |
|------|---------|
| X_train.csv / X_test.csv | 19-feature train and test sets |
| y_train.csv / y_test.csv | Encoded integer labels |
| feature_contract.pkl/.json | Ordered list of 19 feature names |
| label_encoder.pkl | Converts integer predictions → class names |
| median_imputation.json | Fallback values for inf/NaN at inference time |
| feature_extraction_reference.json | Documents how each feature maps to Suricata eve.json |

## 1. Libraries

In [ ]:
import os
import glob
import json
import joblib        # Saving/loading model artifacts (.pkl files)
import warnings
warnings.filterwarnings('ignore')

import pandas as pd  # Data manipulation — loading CSVs, filtering, renaming
import numpy as np   # Numerical operations — np.where, np.inf, np.nan
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder      # Encode class names as integers
from sklearn.model_selection import train_test_split # Stratified train/test split

sns.set(style='darkgrid')
print('Libraries loaded successfully.')

## 2. Configuration

In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ─────────────────────────────────
# Should point to the folder containing all 8 CICIDS2017 CSV files
DATASET_PATH = r"C:/datamine/project/MachineLearningCVE/*.csv"
# ──────────────────────────────────────────────────────────────────────────────

RANDOM_STATE = 42    # Fixed seed — ensures reproducible train/test split
TEST_SIZE    = 0.20  # Hold out 20% for evaluation (504,463 flows)

print(f'Dataset path: {DATASET_PATH}')

## 3. Feature Mapping — CICIDS2017 → Suricata eve.json

**Why this mapping matters:**
The CICIDS2017 dataset uses different column names than Suricata's EVE JSON output.
For example, CICIDS2017 calls forward packets `total_fwd_packets`, while Suricata
calls them `pkts_toserver`. This mapping renames the CICIDS2017 columns to match
what extractor.py will extract from live Suricata traffic.

**The feature contract:**
Only features that can be directly extracted from Suricata's EVE JSON flow records
are included. Features requiring packet-level timing data (e.g. IAT mean, packet
length variance) are NOT available from Suricata's aggregated flow output and are
excluded. This constraint is what limits us to 19 features.

In [ ]:
# Maps CICIDS2017 column name → Suricata eve.json field name
# Only columns that have a direct equivalent in Suricata's flow events
CICIDS_TO_SURICATA = {
    'destination_port'            : 'destination_port',   # Target port (22=SSH, 80=HTTP…)
    'flow_duration'               : 'flow_duration',      # Duration in MICROSECONDS
    'total_fwd_packets'           : 'pkts_toserver',      # Packets client → server
    'total_backward_packets'      : 'pkts_toclient',      # Packets server → client
    'total_length_of_fwd_packets' : 'bytes_toserver',     # Bytes client → server
    'total_length_of_bwd_packets' : 'bytes_toclient',     # Bytes server → client
    'flow_bytes/s'                : 'flow_bytes_per_sec', # Throughput (bytes per second)
    'flow_packets/s'              : 'flow_pkts_per_sec',  # Packet rate
    'down/up_ratio'               : 'down_up_ratio',      # pkts_toclient / pkts_toserver
    'fin_flag_count'              : 'fin_flag',           # TCP FIN present? (0/1)
    'syn_flag_count'              : 'syn_flag',           # TCP SYN present? (0/1)
    'psh_flag_count'              : 'psh_flag',           # TCP PSH present? (0/1)
    'ack_flag_count'              : 'ack_flag',           # TCP ACK present? (0/1)
    'fwd_psh_flags'               : 'tcp_flags_client',   # Combined client flags (hex→int)
}
# NOTE: bwd_psh_flags would map to tcp_flags_server but it was constant
# (always 0) across the entire CICIDS2017 dataset → dropped in Step 8

# These 5 features are computed FROM the mapped columns above
# They don't exist directly in either CICIDS2017 or Suricata — we derive them
DERIVED_FEATURES = [
    'total_bytes',       # bytes_toserver + bytes_toclient
    'total_packets',     # pkts_toserver + pkts_toclient
    'fwd_bytes_per_pkt', # bytes_toserver / pkts_toserver — avg upstream packet size
    'bwd_bytes_per_pkt', # bytes_toclient / pkts_toclient — avg downstream packet size
    'bytes_ratio',       # bytes_toserver / total_bytes — what fraction went upstream?
]

# Final ordered feature list: 14 direct + 5 derived = 19 total
ALL_FEATURES = list(CICIDS_TO_SURICATA.values()) + DERIVED_FEATURES

print(f'Total features: {len(ALL_FEATURES)}')
for i, f in enumerate(ALL_FEATURES, 1):
    print(f'  {i:2d}. {f}')

## 4. Label Grouping

**Why we merge and drop classes:**

**Merged into DoS:**
DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest, and DDoS are all merged
into a single 'DoS' class. At the flow level (with only 19 Suricata features), all
five variants share the same signature: high packet rates, large byte volumes, and
asymmetric flow ratios. The 19 features cannot distinguish between them.

**Merged into Brute Force:**
FTP-Patator and SSH-Patator are merged. Both are credential-guessing attacks with
identical flow-level signatures — many repeated small connections to the same port.
The only difference is the destination port (21 vs 22), which is captured by
`destination_port` anyway.

**Merged into Web Attack:**
Web Attack – Brute Force, XSS, and SQL Injection are merged. All three ride inside
normal HTTP flows and are indistinguishable at the flow level without payload inspection.

**Dropped:**
- Heartbleed: only 11 samples — after 80/20 split only ~9 training samples remain
- Infiltration: only 36 samples — too few for reliable learning

In [ ]:
ATTACK_MAP = {
    'BENIGN'                     : 'BENIGN',
    'DoS Hulk'                   : 'DoS',      # HTTP flood — application layer
    'DoS GoldenEye'              : 'DoS',      # HTTP keep-alive flood
    'DoS slowloris'              : 'DoS',      # Slow connection exhaustion
    'DoS Slowhttptest'           : 'DoS',      # Slow body attack
    'DDoS'                       : 'DoS',      # Distributed flood — merged: same flow signature
    'PortScan'                   : 'PortScan',
    'FTP-Patator'                : 'Brute Force',  # FTP credential guessing (port 21)
    'SSH-Patator'                : 'Brute Force',  # SSH credential guessing (port 22)
    'Bot'                        : 'Bot',
    'Web Attack - Brute Force'   : 'Web Attack',   # HTTP login brute force
    'Web Attack - XSS'           : 'Web Attack',   # Cross-site scripting
    'Web Attack - SQL Injection' : 'Web Attack',   # SQL injection
    # Heartbleed (11 samples)  — dropped: too few for reliable learning
    # Infiltration (36 samples) — dropped: too few for reliable learning
}

print(f'Final classes ({len(set(ATTACK_MAP.values()))}): {sorted(set(ATTACK_MAP.values()))}')

## 5. Load Dataset

In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ─────────────────────────────────
DATASET_PATH = r"C:/Users/msfe-gaza-ict-sup/Desktop/Latest/Abdallah/IDS-ML-project/Dataset/CICIDS2017/CSVs/MachineLearningCVE/*.csv"
# ──────────────────────────────────────────────────────────────────────────────

# Find all 8 CICIDS2017 CSV files matching the glob pattern
file_paths = sorted(glob.glob(DATASET_PATH))
print(f'Found {len(file_paths)} CSV files:')
for fp in file_paths:
    print(f'  {os.path.basename(fp)}')

# Load each CSV into a separate DataFrame
# low_memory=False prevents pandas from guessing column types mid-file
# which would cause mixed-type errors in some CICIDS2017 files
data_list = [pd.read_csv(fp, low_memory=False) for fp in file_paths]

print('\nData dimensions:')
for i, df in enumerate(data_list, 1):
    print(f'  File {i}: {df.shape[0]:,} rows x {df.shape[1]} cols')
# Expected total: 2,830,743 rows across all 8 files

## 6. Merge & Standardize Columns

In [ ]:
# Strip whitespace from column names BEFORE merging
# CICIDS2017 CSVs have leading/trailing spaces in column names
# (e.g. ' Flow Duration' instead of 'Flow Duration')
for df in data_list:
    df.columns = df.columns.str.strip()

# Concatenate all 8 DataFrames into one big dataset
# ignore_index=True resets the row index to 0,1,2,... across all files
data = pd.concat(data_list, ignore_index=True)
del data_list   # Free memory — we no longer need the individual DataFrames

# Standardize column names: lowercase + replace spaces with underscores
# This makes programmatic access consistent: data['flow_duration'] not data['Flow Duration']
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print(f'Merged shape: {data.shape[0]:,} rows x {data.shape[1]} cols')
# Expected: 2,830,743 rows x 79 cols

## 7. Fix Label Encoding Bugs

Some CICIDS2017 CSV files have corrupted 'Web Attack' labels due to encoding
issues during CSV serialization. For example, 'Web Attack – XSS' may appear
as 'Web Attack  XSS' (with a Windows-1252 dash instead of a normal hyphen).
This function normalises all Web Attack labels to consistent strings
regardless of the encoding artifact.

In [ ]:
def fix_label(label):
    """Normalise corrupted Web Attack label strings.
    
    Matches on lowercase keywords to handle any encoding variant.
    Falls back to 'Web Attack - Brute Force' for unknown web attack types.
    """
    label = str(label).strip()
    if 'web attack' in label.lower():
        if 'brute' in label.lower():
            return 'Web Attack - Brute Force'
        elif 'xss' in label.lower():
            return 'Web Attack - XSS'
        elif 'sql' in label.lower():
            return 'Web Attack - SQL Injection'
        else:
            return 'Web Attack - Brute Force'   # Safe fallback
    return label   # All other labels returned unchanged

# Apply the fix to every row in the label column
data['label'] = data['label'].apply(fix_label)

print('Unique labels after fix:')
for lbl in sorted(data['label'].unique()):
    print(f'  {lbl}')
# Should show 15 clean label strings with no encoding artifacts

## 8. Drop Constant & Duplicate Columns

**Constant columns** (zero variance) carry no information — every row has the
same value so the model can never learn anything from them. We drop them to
reduce noise and memory usage.

**Known constant columns in CICIDS2017:**
`bwd_psh_flags`, `bwd_urg_flags`, and 6 bulk rate features — all zero across
the entire dataset.

**Note about bwd_psh_flags:**
This column would have mapped to `tcp_flags_server` in our feature contract,
but because it's constant it gets dropped here. This is why we end up with
19 features instead of the original 20 in our feature mapping.

In [ ]:
cols_to_drop = []

# Find columns with only one unique value (constant = useless for ML)
for col in data.columns:
    if col == 'label':
        continue   # Never drop the target column
    if data[col].nunique() == 1:
        print(f'  Constant: {col}')
        cols_to_drop.append(col)

# fwd_header_length.1 is a duplicate of fwd_header_length — a known CICIDS2017 bug
if 'fwd_header_length.1' in data.columns:
    cols_to_drop.append('fwd_header_length.1')

data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True)

# Remove exact duplicate rows — same flow appearing more than once
# (CICIDS2017 has 308,381 duplicates from overlapping capture windows)
n_dups = data.duplicated().sum()
print(f'Duplicate rows: {n_dups:,}')
data.drop_duplicates(inplace=True)

print(f'Shape after cleanup: {data.shape[0]:,} x {data.shape[1]}')
# Expected: ~2,522,362 rows x 70 cols

## 9. Drop Heartbleed & Infiltration, Apply Label Grouping

In [ ]:
# Drop rows whose labels are NOT in ATTACK_MAP
# This filters out Heartbleed (11 rows) and Infiltration (36 rows)
# We use isin() on the ATTACK_MAP keys — any label not in the map gets removed
before = len(data)
data = data[data['label'].isin(ATTACK_MAP.keys())].copy()
dropped = before - len(data)
print(f'Dropped {dropped} rows (Heartbleed + Infiltration)')

# Map original fine-grained labels to our 6 consolidated classes
# e.g. 'DoS Hulk' → 'DoS', 'FTP-Patator' → 'Brute Force'
data['attack_grouped'] = data['label'].map(ATTACK_MAP)

print(f'\nRemaining rows: {len(data):,}')
print(f'\nClass distribution after grouping:')
dist = data['attack_grouped'].value_counts()
for cls, cnt in dist.items():
    print(f'  {cls:<15} {cnt:>8,}   {cnt/len(data)*100:.2f}%')
# Expected: BENIGN=2,096,484 (83.12%), DoS=321,764 (12.76%), etc.

## 10. Select & Rename Features

This step:
1. Renames the 14 CICIDS2017 columns to their Suricata-aligned names
2. Computes the 5 derived features
3. Selects only the 19 features in our contract — drops everything else

**Critical: zero-division fallbacks must match extractor.py exactly.**
If training uses 0.0 as the fallback for bytes_ratio but extractor.py uses 0.5,
the model will see different distributions at training time vs inference time.

In [ ]:
# Step 1: Rename CICIDS2017 columns to Suricata-aligned names
# (e.g. 'total_fwd_packets' → 'pkts_toserver')
data.rename(columns=CICIDS_TO_SURICATA, inplace=True)

# Step 2: Compute the 5 derived features
# CRITICAL: fallback values here must match extractor.py exactly
# If they differ, the model trains on a different distribution than it sees at runtime

data['total_bytes']   = data['bytes_toserver'] + data['bytes_toclient']
data['total_packets'] = data['pkts_toserver']  + data['pkts_toclient']

# np.where(condition, value_if_true, value_if_false)
# Avoids division by zero — matches the if/else guards in extractor.py
data['fwd_bytes_per_pkt'] = np.where(data['pkts_toserver'] > 0,
                                      data['bytes_toserver'] / data['pkts_toserver'], 0)
data['bwd_bytes_per_pkt'] = np.where(data['pkts_toclient'] > 0,
                                      data['bytes_toclient'] / data['pkts_toclient'], 0)

# bytes_ratio fallback is 0.5, NOT 0.0
# Reason: a flow with zero total bytes is perfectly symmetric by definition,
# and 0.5 represents symmetry. Using 0.0 would falsely imply all traffic went server→client.
data['bytes_ratio'] = np.where(data['total_bytes'] > 0,
                                data['bytes_toserver'] / data['total_bytes'], 0.5)

# Step 3: Select only the 19 features in contract order
# (drops all other CICIDS2017 columns we don't need)
available = [f for f in ALL_FEATURES if f in data.columns]
missing   = [f for f in ALL_FEATURES if f not in data.columns]

if missing:
    print(f'WARNING — missing features: {missing}')

X_all = data[available].copy()   # Feature matrix
y_raw = data['attack_grouped'].copy()   # Target labels (string class names)

print(f'Feature matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} cols')
print(f'Features: {available}')

## 11. Handle Missing & Infinite Values

**Why inf values occur:**
`flow_bytes/s` and `flow_packets/s` in CICIDS2017 are computed as rate/duration.
When a flow has duration = 0 (an instantaneous connection), this produces inf.
We replace inf with NaN, then impute with the column median.

**Why median (not mean)?**
The mean is heavily skewed by extreme values (DoS flows have very high byte rates).
The median is robust — it represents the "typical" value even in imbalanced data.

**Why save the medians?**
At inference time (in detector.py), the same inf/NaN situation can occur from live
Suricata flows. We must apply the EXACT SAME median values computed here —
not recompute them — to keep the feature distributions consistent.

In [ ]:
# Replace inf and -inf with NaN so pandas can handle them
# inf arises from flow_bytes/s and flow_pkts/s when flow_duration = 0
X_all.replace([np.inf, -np.inf], np.nan, inplace=True)

missing = X_all.isna().sum()
affected = missing[missing > 0]   # Only columns that have NaN

if len(affected) > 0:
    print('Columns with missing values:')
    display(pd.concat([affected, (affected / len(X_all) * 100).round(3)], axis=1)
            .rename(columns={0: 'Count', 1: '%'})
            .style.background_gradient(cmap='Reds'))

    # Impute each affected column with its median value
    # Computed on the FULL dataset (before split) to match runtime behavior
    # where we don't know train/test boundaries
    median_vals = {}
    for col in affected.index:
        med = X_all[col].median()
        X_all[col].fillna(med, inplace=True)
        median_vals[col] = med
        print(f'  Imputed {col} with median = {med:.4f}')
else:
    print('No missing values.')
    median_vals = {}   # Empty dict — still saved to median_imputation.json

## 12. Class Distribution Visualization

Two views of the same data:
- **Linear scale** — shows the dominance of BENIGN traffic (83.12%)
- **Log scale** — reveals the minority classes (Bot, Web Attack) which would be
  invisible on a linear scale

This severe imbalance is why we use class_weight='balanced' rather than SMOTE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dist = y_raw.value_counts()

# Linear scale — shows BENIGN dominates everything else
axes[0].bar(dist.index, dist.values, color=sns.color_palette('Set2', len(dist)))
axes[0].set_title('Class distribution (linear)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Log scale — minority classes (Bot=1953, Web Attack=2143) become visible
axes[1].bar(dist.index, dist.values, color=sns.color_palette('Set2', len(dist)))
axes[1].set_yscale('log')
axes[1].set_title('Class distribution (log scale)')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFinal class counts:')
for cls, cnt in dist.items():
    print(f'  {cls:<15} {cnt:>8,}   {cnt/len(y_raw)*100:.2f}%')

## 13. Encode Labels

scikit-learn's RandomForestClassifier works with integer labels internally.
LabelEncoder converts our 6 string class names to integers 0–5.
The mapping is alphabetical:
- 0 = BENIGN
- 1 = Bot
- 2 = Brute Force
- 3 = DoS
- 4 = PortScan
- 5 = Web Attack

The encoder is saved to label_encoder.pkl so detector.py can convert
integer model predictions back to human-readable class names.

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)   # Converts class names → integers

print(f'Classes ({len(le.classes_)}): {list(le.classes_)}')
print(f'Encoded values: {list(range(len(le.classes_)))}')

# Binary label: 0=BENIGN, 1=any attack
# Used for binary-level performance evaluation (overall attack detection rate)
data['attack_binary'] = (y_raw != 'BENIGN').astype(int)

## 14. Train / Test Split (Stratified)

**80% training, 20% test** — a standard split giving:
- Training: 2,017,852 samples
- Test: 504,463 samples (the held-out evaluation set)

**Stratified split** (`stratify=y_encoded`) ensures each class maintains its
original proportion in both the training and test sets. Without stratification,
minority classes (Bot=1953, Web Attack=2143) could be severely under-represented
in the test set, making evaluation unreliable.

**random_state=42** ensures the split is reproducible — running this notebook
again always produces the exact same train/test partition.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded       # Preserves class ratios in both splits
)

print(f'Train: {X_train.shape[0]:,} rows')
print(f'Test:  {X_test.shape[0]:,} rows')

# Verify the split maintained class proportions
print(f'\nTrain class distribution:')
train_dist = pd.Series(y_train).value_counts()
for enc, cnt in train_dist.items():
    cls = le.inverse_transform([enc])[0]
    print(f'  {cls:<15} {cnt:>8,}   {cnt/len(y_train)*100:.2f}%')

## 15. Class Weights (for Random Forest)

No SMOTE — using `class_weight='balanced'` in the classifier instead.

**Why not SMOTE?**
- SMOTE interpolates synthetic samples between real flows
- Binary TCP flags (fin, syn, psh, ack) are 0 or 1 — interpolation produces
  meaningless values like 0.4 or 0.7 that don't represent real network flags
- Flow rate features near zero can go negative through SMOTE interpolation —
  physically impossible for packet counts or byte volumes
- `class_weight='balanced'` achieves mathematically equivalent class reweighting
  without modifying the training data at all

**How class_weight='balanced' works:**
The formula is: weight = total_samples / (n_classes × class_count)
This means Bot and Web Attack receive ~200x more weight than BENIGN during training,
giving the model a strong incentive to correctly classify minority classes.

In [ ]:
from collections import Counter

class_counts = Counter(y_train)
total = sum(class_counts.values())
n_classes = len(class_counts)

# Compute and display the weights that Random Forest will apply internally
# Formula: weight_c = total / (n_classes * count_c)
print('Class weights that Random Forest will apply internally:')
print(f'{"Class":<20} {"Samples":>10} {"Weight":>10}')
print('-' * 42)
for enc in sorted(class_counts.keys()):
    cls    = le.inverse_transform([enc])[0]
    cnt    = class_counts[enc]
    weight = total / (n_classes * cnt)
    print(f'{cls:<20} {cnt:>10,} {weight:>10.3f}')

print('\nHigher weight = minority class gets more influence during tree building.')
# Bot and Web Attack should show weights ~200x higher than BENIGN

## 16. Feature Selection — Keep Final 19

In [ ]:
# After Step 8 dropped bwd_psh_flags (constant column), ALL_FEATURES already
# contains exactly 19 features. No further dropping needed.
FINAL_FEATURES = ALL_FEATURES

# Select only the 19 final features from both splits
# This ensures no extra columns accidentally slip into the training data
X_train_final = X_train[FINAL_FEATURES]
X_test_final  = X_test[FINAL_FEATURES]

print(f'Final feature count: {len(FINAL_FEATURES)}')
for i, f in enumerate(FINAL_FEATURES, 1):
    print(f'  {i:2d}. {f}')

## 17. Feature Statistics

Descriptive statistics for the 19 training features.
The 'skew' column reveals heavily right-skewed features (high skew = DoS/attack
flows pull the mean far above the median). Random Forest is unaffected by skew
because it uses threshold splits, not distances.

In [ ]:
# Compute standard statistics (count, mean, std, min, quartiles, max)
# plus skewness for each feature
stats = X_train_final.describe().T
stats['skew'] = X_train_final.skew()

# Colour-gradient on mean and std to spot high-variance features easily
display(stats.style.background_gradient(cmap='Blues', subset=['mean', 'std']))

## 18. Correlation Heatmap

Checks for highly correlated features (|r| > 0.9).
High correlation is expected and acceptable here — for example,
`total_bytes` is the sum of `bytes_toserver` and `bytes_toclient`,
so they are naturally correlated.

Random Forest handles correlated features natively by randomly selecting
a subset of features at each split (max_features='sqrt'). Unlike linear models
or PCA pipelines, we do NOT need to drop correlated features.

In [ ]:
corr = X_train_final.corr().abs()   # Absolute correlation matrix

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, cbar=True,
            linewidths=0.5, annot=False)
plt.title('Feature Correlation Heatmap — 19 Suricata Features', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# List all pairs with correlation > 0.9
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if corr.iloc[i, j] > 0.9:
            high_corr_pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

print(f'Highly correlated pairs (>0.9): {len(high_corr_pairs)}')
for a, b, v in sorted(high_corr_pairs, key=lambda x: -x[2]):
    print(f'  {a} — {b}: {v:.3f}')
print('\nNote: High correlation is expected (e.g. total_bytes vs bytes_toserver).')
print('Random Forest handles correlated features natively — no action needed.')

## 19. Save All Outputs

Saves everything needed by the model training notebook and the deployment pipeline.

**Files for model training:**
- X_train.csv, X_test.csv, y_train.csv, y_test.csv

**Files for deployment (used by detector.py at inference time):**
- feature_contract.pkl — tells detector.py which 19 features to use and in what order
- label_encoder.pkl — converts integer predictions (0–5) back to class names
- median_imputation.json — fallback values for inf/NaN at runtime

**Documentation file:**
- feature_extraction_reference.json — documents exactly how each feature maps to Suricata

In [ ]:
# ── Train/Test splits ─────────────────────────────────────────────────────────
X_train_final.to_csv('X_train.csv', index=False)
X_test_final.to_csv('X_test.csv',   index=False)

# ── Labels ────────────────────────────────────────────────────────────────────
pd.Series(y_train, name='label').to_csv('y_train.csv', index=False)
pd.Series(y_test,  name='label').to_csv('y_test.csv',  index=False)

# Extra label files for reference/analysis (not used by training)
y_raw.to_csv('grouped_labels.csv')
data['attack_binary'].to_csv('binary_labels.csv')
data['label'].to_csv('original_labels.csv')

# ── Feature contract ──────────────────────────────────────────────────────────
# Saved as both .pkl (used by detector.py) and .json (human readable)
joblib.dump(FINAL_FEATURES, 'feature_contract.pkl')
with open('feature_contract.json', 'w') as f:
    json.dump(FINAL_FEATURES, f, indent=2)

# ── Label encoder ─────────────────────────────────────────────────────────────
joblib.dump(le, 'label_encoder.pkl')

# ── Median imputation values ──────────────────────────────────────────────────
# CRITICAL: detector.py loads this file and applies these exact medians at runtime
# to replace any inf/NaN values it encounters in live Suricata flows
with open('median_imputation.json', 'w') as f:
    json.dump(median_vals, f, indent=2)

# ── Feature extraction reference (documentation only) ─────────────────────────
# A human-readable reference showing exactly how each feature should be
# extracted from Suricata EVE JSON. Used to verify extractor.py is correct.
extraction_ref = {
    'description': 'How to extract each feature from Suricata eve.json',
    'n_features': len(FINAL_FEATURES),
    'zero_division_fallbacks': {
        'flow_bytes_per_sec' : '0   (if flow_duration == 0)',
        'flow_pkts_per_sec'  : '0   (if flow_duration == 0)',
        'down_up_ratio'      : '0   (if pkts_toserver == 0)',
        'fwd_bytes_per_pkt'  : '0   (if pkts_toserver == 0)',
        'bwd_bytes_per_pkt'  : '0   (if pkts_toclient == 0)',
        'bytes_ratio'        : '0.5 (if total_bytes == 0) — symmetric flow assumption',
    },
    'features': {
        'destination_port'   : "event['dest_port']",
        'flow_duration'      : "(parse(flow['end']) - parse(flow['start'])).total_seconds() * 1e6",
        'pkts_toserver'      : "flow['pkts_toserver']",
        'pkts_toclient'      : "flow['pkts_toclient']",
        'bytes_toserver'     : "flow['bytes_toserver']",
        'bytes_toclient'     : "flow['bytes_toclient']",
        'flow_bytes_per_sec' : "total_bytes / flow_duration_seconds  [0 if duration=0]",
        'flow_pkts_per_sec'  : "total_packets / flow_duration_seconds [0 if duration=0]",
        'down_up_ratio'      : "pkts_toclient / pkts_toserver [0 if pkts_toserver=0]",
        'fin_flag'           : "1 if tcp.get('fin') else 0",
        'syn_flag'           : "1 if tcp.get('syn') else 0",
        'psh_flag'           : "1 if tcp.get('psh') else 0",
        'ack_flag'           : "1 if tcp.get('ack') else 0",
        'tcp_flags_client'   : "int(tcp.get('tcp_flags_ts', '00'), 16)",
        'total_bytes'        : "bytes_toserver + bytes_toclient",
        'total_packets'      : "pkts_toserver  + pkts_toclient",
        'fwd_bytes_per_pkt'  : "bytes_toserver / pkts_toserver  [0 if pkts_toserver=0]",
        'bwd_bytes_per_pkt'  : "bytes_toclient / pkts_toclient  [0 if pkts_toclient=0]",
        'bytes_ratio'        : "bytes_toserver / total_bytes    [0.5 if total_bytes=0]",
    }
}
with open('feature_extraction_reference.json', 'w') as f:
    json.dump(extraction_ref, f, indent=2)

print('All outputs saved:')
print(f'  X_train.csv / X_test.csv          — {X_train_final.shape[1]} features')
print(f'  y_train.csv / y_test.csv')
print(f'  feature_contract.json/.pkl         — {len(FINAL_FEATURES)} features')
print(f'  label_encoder.pkl                  — {len(le.classes_)} classes: {list(le.classes_)}')
print(f'  median_imputation.json             — fallback values for detector.py')
print(f'  feature_extraction_reference.json')
print()
print(f'Preprocessing complete — {len(FINAL_FEATURES)} features, {len(le.classes_)} classes.')
print('Ready for model_training_final.ipynb')